# Notebook 02 — Full §4.71a + §4.1–§4.31 CFR Ingestion

Slice 3 ingested a single Diagnostic Code (DC 5260) as a tracer. This notebook broadens to the full v1 CFR scope:

1. **Auto-discover** every Diagnostic Code in §4.71a (musculoskeletal).
2. **Bulk ingest** all of them via the shared validate-then-write pipeline.
3. **Ingest a couple of general-provision Rules** from §4.1–§4.31 (e.g. §4.14 Pyramiding, §4.26 Bilateral Factor) as `:CFR:Rule` nodes.
4. **Enrich Anatomy** with body_system, side, parent_anatomy, and `:PAIRED_WITH` edges between left/right.
5. **Query** the resulting graph.

**Prereqs**

- `make up` — Neo4j running
- `OPENAI_API_KEY` set in `.env`
- The §4.71a XML cached at `data/ecfr_cache/2025-01-01-t38-p4-s4.71a.xml` (slice 3 fetched it).

**Cost note:** Cell 3 makes one OpenAI call per Diagnostic Code (~160 calls). The XML cache prevents re-fetching but LLM calls are not cached.

## 1. Discover every Diagnostic Code in §4.71a

`discover_dc_codes` scans the section's `<TR>` rows; each row whose first `<TD>` begins with a 4-digit token is a Diagnostic Code header. Hand-curated lists rot — this is the source of truth.

In [ ]:
from va_agent.ingestion.discovery import discover_dc_codes
from va_agent.ingestion.ecfr_client import fetch_section_xml

xml_text, cache_path = fetch_section_xml(section='4.71a')
dc_codes = discover_dc_codes(xml_text)
print(f'Found {len(dc_codes)} Diagnostic Codes in §4.71a')
print('First 10:', dc_codes[:10])
print('Last 5: ', dc_codes[-5:])

## 2. Bulk-ingest §4.71a

`ingest_section_full` orchestrates: fetch → discover → for each DC (extract → validate → write) → anatomy enrichment → cross-reference resolution.

Failed validations are diverted to `data/review_queue.jsonl` instead of the graph — inspect that file afterwards to see what the LLM couldn't be trusted on.

In [ ]:
from va_agent.graph.driver import GraphDriver
from va_agent.ingestion.cfr import ingest_section_full

with GraphDriver.from_env() as driver:
    report = ingest_section_full(section='4.71a', driver=driver)

print(f'attempted: {len(report.dc_codes_attempted)}')
print(f'written:   {len(report.dc_codes_written)}')
print(f'failed:    {len(report.dc_codes_failed)}')
for code, errs in report.dc_codes_failed[:5]:
    print(f'  {code}: {errs}')
for w in report.warnings[-2:]:  # the anatomy + xref post-pass summaries
    print(' ', w)

## 3. Ingest a couple of general-provision Rules

§4.1–§4.31 do not contain Diagnostic Codes; they're prose rules that condition how Rating Percentages are derived and combined. `ingest_rule_section` writes each as a `:CFR:Rule` node. The most load-bearing for v1 are:

- **§4.14 — Pyramiding** (no double-rating the same symptom)
- **§4.25 — Combined Ratings Table** (how Rating Percentages combine; v1 = text only, structured table values are a TODO)
- **§4.26 — Bilateral Factor** (the ~10% uplift for matching-limb Disabilities)

In [ ]:
from va_agent.ingestion.cfr import ingest_rule_section

with GraphDriver.from_env() as driver:
    for section in ['4.14', '4.25', '4.26', '4.40', '4.45', '4.59']:
        sub = ingest_rule_section(section=section, driver=driver)
        print(f'§{section}: written={sub.rules_written} failed={sub.rules_failed}')

## 4. Query the resulting graph

Count what landed, peek at the bilateral structure for the knees, and read one Rule's text.

In [ ]:
with GraphDriver.from_env() as driver:
    counts = driver.cfr_read(
        '''
        MATCH (dc:CFR:DiagnosticCode) WITH count(dc) AS dcs
        MATCH (r:CFR:Rule)            WITH dcs, count(r) AS rules
        MATCH (a:CFR:Anatomy)         WITH dcs, rules, count(a) AS anatomy
        MATCH (m:CFR:Measurement)     RETURN dcs, rules, anatomy, count(m) AS measurements
        '''
    )
    print('Graph counts:', counts[0])

    print()
    print('Bilateral knee structure:')
    for row in driver.cfr_read(
        '''
        MATCH (l:CFR:Anatomy {parent_anatomy: 'knee', side: 'left'})-[:PAIRED_WITH]->(r:CFR:Anatomy {parent_anatomy: 'knee', side: 'right'})
        RETURN l.name AS left, r.name AS right, l.body_system AS body_system
        '''
    ):
        print(' ', row)

    print()
    print('§4.14 (Pyramiding):')
    for row in driver.cfr_read(
        "MATCH (r:CFR:Rule {id: 'pyramiding'}) RETURN r.name AS name, r.text AS text"
    ):
        print(f'  {row["name"]}')
        print(f'  {row["text"][:300]}...')

## 5. Review queue

Anything the validator rejected sits in `data/review_queue.jsonl` one entry per line. Inspect there to triage DCs whose CFR text the LLM couldn't faithfully convert — these become the targets for prompt-tuning or hand-overrides in slice 5+.

```bash
wc -l data/review_queue.jsonl
tail -n 5 data/review_queue.jsonl | jq .
```